In [1]:
import polars as pl
from datasetsforecast.hierarchical import HierarchicalData
from IPython.display import display

import sys
sys.path.append('../../04_utils')

from utils import continuity_fix
from visualization import plot_time_series

In [2]:
# Set Path variables
raw_data_path = '../../01_data/01_raw'

## Data Downloading

In [3]:
# # Download the data for the demo
# HierarchicalData.download(raw_data_path)

## Data Loading

In [4]:
# Load and check the downloaded data
# Data contains the TS dataframe and the hierarchical matrix, for this part of the study only 
# the TS dataframe is relevant.
data_df = HierarchicalData.load(raw_data_path, 'TourismLarge', cache=True)

data_df

(       unique_id          ds             y
 0       TotalAll  1998-01-01  45151.071280
 1       TotalAll  1998-02-01  17294.699551
 2       TotalAll  1998-03-01  20725.114184
 3       TotalAll  1998-04-01  25388.612353
 4       TotalAll  1998-05-01  20330.035211
 ...          ...         ...           ...
 126535    GBDOth  2016-08-01      0.000000
 126536    GBDOth  2016-09-01      0.000000
 126537    GBDOth  2016-10-01      0.000000
 126538    GBDOth  2016-11-01      0.000000
 126539    GBDOth  2016-12-01      0.000000
 
 [126540 rows x 3 columns],
           AAAHol  AAAVis  AAABus  AAAOth  AABHol  AABVis  AABBus  AABOth  \
 TotalAll     1.0     1.0     1.0     1.0     1.0     1.0     1.0     1.0   
 AAll         1.0     1.0     1.0     1.0     1.0     1.0     1.0     1.0   
 BAll         0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 CAll         0.0     0.0     0.0     0.0     0.0     0.0     0.0     0.0   
 DAll         0.0     0.0     0.0     0.0     0.0     0.0

In [5]:
# Assign the TS dataframe its own variable
ts_df = pl.from_pandas(data_df[0])

# As our date column is in a str format, we need to transform it into a proper date format.
# According to Nixtla's documentation, this data has a monthly frequency.
ts_df = ts_df.with_columns(pl.col('ds').str.to_date(format='%Y-%m-%d').alias('ds'))

ts_df.head()

unique_id,ds,y
str,date,f64
"""TotalAll""",1998-01-01,45151.07128
"""TotalAll""",1998-02-01,17294.699551
"""TotalAll""",1998-03-01,20725.114184
"""TotalAll""",1998-04-01,25388.612353
"""TotalAll""",1998-05-01,20330.035211


## Unique TS Study

In [6]:
# Look for null values in unique_id
ts_df.filter(pl.col('unique_id').is_null())

unique_id,ds,y
str,date,f64


In [7]:
# Look for null values in y
ts_df.filter(pl.col('y').is_null())

unique_id,ds,y
str,date,f64


In [8]:
# Look for null values in ds
ts_df.filter(pl.col('ds').is_null())

unique_id,ds,y
str,date,f64


No null values for any of the relevant columns

In [9]:
# Create a dataframe for studying each individual time-series.
uid_info_df = ts_df.group_by('unique_id').agg(pl.count('ds').alias('n_entries'), pl.min('ds').alias('min_date'),
                                              pl.max('ds').alias('max_date'))

uid_info_df

unique_id,n_entries,min_date,max_date
str,u32,date,date
"""DCBOth""",228,1998-01-01,2016-12-01
"""CBOth""",228,1998-01-01,2016-12-01
"""CDAll""",228,1998-01-01,2016-12-01
"""FAAll""",228,1998-01-01,2016-12-01
"""AEDBus""",228,1998-01-01,2016-12-01
…,…,…,…
"""BDFOth""",228,1998-01-01,2016-12-01
"""TotalVis""",228,1998-01-01,2016-12-01
"""EBABus""",228,1998-01-01,2016-12-01


In [10]:
# Check the 3 main columns to see if all of the time-series have the same n_entries and start and end at the same time.
uid_info_df['n_entries'].unique()

n_entries
u32
228


In [11]:
uid_info_df['min_date'].unique()

min_date
date
1998-01-01


In [12]:
uid_info_df['max_date'].unique()

max_date
date
2016-12-01


It seems that all of the time-series are equalized when it comes to amount of points and their time period. Let's check for holes/missing values in our data.

In [13]:
# Since all time-series in our data have the same start and ending points 
# we'll store them in variables for this part of the analysis
min_date = uid_info_df['min_date'].unique()[0]
max_date = uid_info_df['max_date'].unique()[0]


In [14]:
# We run the continuity_fix function implemented in the utils.py file through all the unique time-series
# to check for holes and fill them.
uids = ts_df['unique_id'].unique()

for uid in uids:
    ts = ts_df.filter(pl.col('unique_id') == uid)
    if uid == uids[0]:
        fixed_ts_df = continuity_fix(ts, min_date, max_date, 'MS', 'unique_id', )
    else:
        fixed_ts_df = pl.concat([fixed_ts_df, continuity_fix(ts, min_date, max_date, 'MS', 'unique_id', )])

fixed_ts_df

unique_id,ds,y,hole
str,date,f64,i32
"""AFAll""",1998-01-01,612.447811,0
"""AFAll""",1998-02-01,471.262833,0
"""AFAll""",1998-03-01,430.198816,0
"""AFAll""",1998-04-01,499.125467,0
"""AFAll""",1998-05-01,338.317473,0
…,…,…,…
"""DAOth""",2016-08-01,52.855307,0
"""DAOth""",2016-09-01,176.640081,0
"""DAOth""",2016-10-01,155.997687,0


In [15]:
# Check if there were any holes in our time-series
fixed_ts_df.group_by('unique_id').agg(pl.sum('hole')).sort('hole', descending= True)

unique_id,hole
str,i32
"""ABHol""",0
"""CCAVis""",0
"""BEVis""",0
"""FABus""",0
"""BDDOth""",0
…,…
"""GABVis""",0
"""GBBVis""",0
"""ADAOth""",0


It seems like our time-series are continuous and consistent when it comes to start and ending date, as such it should be fine to use them as is. Still, for extra insurance we'll visualize them before giving them the final approval seal.

## Time-Series Visualization

In [16]:
# We call the plot_time_series function defined in the visualization.py file in the 04_utils folder
# to plot and observe our time series as a final check of their integrity and to have a way to
# observe time series in the future should it become necessary during development. 
plot_time_series(ts_df.to_pandas(), 'unique_id')

Time series seem to be looking good so far. Due to the sheer amount of them is not realistic to check all of them manually, but should the need arise, this notebook can be used to look at any problematic time series.